<a href="https://colab.research.google.com/github/Lijashree/Devadharshini-agenticAI/blob/main/Agentic_AI_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-text-splitters
!pip install sentence-transformers faiss-cpu chromadb
!pip install groq pypdf pymupdf

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Devadharshini_IT.pdf")
documents = loader.load()

In [ ]:
!pip install faiss-cpu sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
recursive_docs = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(
    recursive_docs,      # 👈 your chunks
    embedding
)

In [ ]:
# For recursive
vectorstore = FAISS.from_documents(recursive_docs, embedding)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
vectorstore.save_local("resume_vector_db")

In [ ]:
query = "What are the skills?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n🔹 Result {i+1}:")
    print(doc.page_content)
    print("-" * 50)

In [ ]:
print("Total vectors stored:", len(vectorstore.index_to_docstore_id))

In [ ]:
def chat_with_resume(query):
    # 🔍 Retrieve relevant chunks
    docs = retriever.invoke(query)

    # 🧠 Combine context
    context = "\n".join([doc.page_content for doc in docs])

    # 🤖 Ask LLM
    response = llm.invoke(
        f"""
        You are a resume assistant.
        Answer only from the given context.

        Context:
        {context}

        Question:
        {query}
        """
    )

    return response.content